# Notebook 1: Exploring the VidOR / VidVRD Dataset

**Goal:** Understand the Video Visual Relation Detection datasets (VidOR and VidVRD), their annotation structure, class taxonomies, and how `data/prepare.py` preprocesses raw annotations into training-ready pickles.

VRDFormer (CVPR 2022) works on two datasets:
- **VidVRD** (ImageNet-VidVRD): 1,000 videos, 35 object classes, 132 predicate classes
- **VidOR** (Video Object Relation): 10,000 videos, 80 object classes, 50 predicate classes

Both datasets annotate `<subject, predicate, object>` triplets across video frames, with object track IDs that persist across frames.

## 1. Setup & Imports

In [1]:
import os
import sys
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Add repo root to path so we can import our modules
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

print(f'Root directory: {ROOT_DIR}')

Root directory: /mnt/c/Users/Samuel Oliveira/Desktop/CS/VRDFormer_VRD


## 2. Explore Class Taxonomies

VRDFormer uses these class files to build prediction heads. Let's inspect each one.

In [ ]:
# --- VidVRD classes ---
vidvrd_dir = ROOT_DIR / 'data' / 'vidvrd'

with open(vidvrd_dir / 'obj.txt', 'r') as f:
    vidvrd_objects = [l.strip() for l in f.readlines()]
with open(vidvrd_dir / 'action.txt', 'r') as f:
    vidvrd_predicates = [l.strip() for l in f.readlines()]

print(f'VidVRD: {len(vidvrd_objects)} object classes, {len(vidvrd_predicates)} predicate classes')
print(f'\nObject classes (first 10): {vidvrd_objects[:10]}')
print(f'\nPredicate classes (first 10): {vidvrd_predicates[:10]}')

In [2]:
# --- VidOR classes ---
vidor_dir = ROOT_DIR / 'data' / 'vidor'

with open(vidor_dir / 'obj.txt', 'r') as f:
    vidor_objects = [l.strip() for l in f.readlines()]
with open(vidor_dir / 'rel.txt', 'r') as f:
    vidor_relations = [l.strip() for l in f.readlines()]
with open(vidor_dir / 'action.txt', 'r') as f:
    vidor_actions = [l.strip() for l in f.readlines()]
with open(vidor_dir / 'spatial.txt', 'r') as f:
    vidor_spatials = [l.strip() for l in f.readlines()]

print(f'VidOR: {len(vidor_objects)} object classes')
print(f'  Relations (action + spatial): {len(vidor_relations)}')
print(f'  Actions only: {len(vidor_actions)}')
print(f'  Spatial only: {len(vidor_spatials)}')
print(f'\nObject classes (first 15): {vidor_objects[:15]}')
print(f'\nSpatial relations: {vidor_spatials}')

VidOR: 80 object classes
  Relations (action + spatial): 50
  Actions only: 50
  Spatial only: 8

Object classes (first 15): ['bread', 'cake', 'dish', 'fruits', 'vegetables', 'backpack', 'camera', 'cellphone', 'handbag', 'laptop', 'suitcase', 'ball/sports_ball', 'bat', 'frisbee', 'racket']

Spatial relations: ['above', 'away', 'behind', 'beneath', 'in_front_of', 'inside', 'next to', 'toward']


**Key insight:** VidVRD has 132 predicates that mix actions and spatial relations. VidOR splits them explicitly into 42 actions + 8 spatials = 50 relations. The model head dimension is `num_verb_classes` which is 132 for VidVRD, 50 for VidOR.

## 3. Explore Raw Annotation Structure

Raw annotations are JSON files with per-frame object trajectories and relation instances. Let's load a sample to understand the structure.

In [4]:
# Configure your dataset path here:
# For VidVRD: set VIDVRD_PATH to where videos/ and annotations/ folders live
# For VidOR: set VIDOR_PATH similarly

DATASET = 'vidor'  # or 'vidor'
DATA_PATH = Path(os.environ.get('VIDVRD_PATH', '/home/zhengsipeng/data/vidvrd'))

print(f'Dataset: {DATASET}')
print(f'Data path: {DATA_PATH}')

Dataset: vidor
Data path: /home/zhengsipeng/data/vidvrd


In [5]:
# Load one annotation file and explore its structure
import glob

anno_files = sorted(glob.glob(str(DATA_PATH / 'annotations' / 'train' / '*.json')))
print(f'Found {len(anno_files)} annotation files')

if len(anno_files) > 0:
    with open(anno_files[0], 'r') as f:
        sample_anno = json.load(f)
    
    print(f'\nTop-level keys: {list(sample_anno.keys())}')
    print(f'Video ID: {sample_anno.get("video_id", "N/A")}')
    print(f'Frame count: {sample_anno.get("frame_count", "N/A")}')
    print(f'FPS: {sample_anno.get("fps", "N/A")}')
    
    # Explore trajectories
    trajectories = sample_anno.get('trajectories', [])
    print(f'\nNumber of object trajectories: {len(trajectories)}')
    if len(trajectories) > 0:
        sample_traj = trajectories[0]
        print(f'Sample trajectory keys: {list(sample_traj.keys())}')
        print(f'  Track ID: {sample_traj.get("tid")}')
        print(f'  Category: {sample_traj.get("category")}')
        print(f'  Number of frames with bbox: {len(sample_traj.get("bbox", {}))}')
    
    # Explore relation instances
    relations = sample_anno.get('relation_instances', [])
    print(f'\nNumber of relation instances: {len(relations)}')
    if len(relations) > 0:
        sample_rel = relations[0]
        print(f'Sample relation keys: {list(sample_rel.keys())}')
        print(f'  Subject TID: {sample_rel.get("subject_tid")}')
        print(f'  Object TID: {sample_rel.get("object_tid")}')
        print(f'  Predicate: {sample_rel.get("predicate")}')
        print(f'  Begin FID: {sample_rel.get("begin_fid")}')
        print(f'  End FID: {sample_rel.get("end_fid")}')
    
    # Subject/object mapping
    print(f'\nSubject/objects mapping:')
    so = sample_anno.get('subject/objects', {})
    print(f'  Number of entities: {len(so)}')
    if len(so) > 0:
        first_key = list(so.keys())[0]
        print(f'  Example: tid={first_key} -> category="{so[first_key]["category"]}"')

Found 0 annotation files


## 4. Running Data Preparation

The function `generate_vrd_annos()` in `data/prepare.py` converts raw JSON annotations into a per-frame pickle format that the PyTorch Dataset can efficiently load. Let's run it programmatically.

In [ ]:
# Import the data preparation functions
sys.path.insert(0, str(ROOT_DIR / 'data'))
from prepare import generate_vrd_annos, get_trainval_frameids

print('Functions imported:')
print('  - generate_vrd_annos(dbname, data_dir) -> creates annotations pickle')
print('  - get_trainval_frameids(dbname, data_dir, split, stage, timestep, minmax_dur) -> creates frame index JSON')

In [ ]:
# Step A: Generate the annotations pickle
# WARNING: This needs to be run from the `data/` directory because prepare.py uses relative paths.
# If you already have data/metadata/<db>_annotations.pkl, skip this step.

import os as _os
_orig_cwd = _os.getcwd()
_os.chdir(str(ROOT_DIR / 'data'))

try:
    # generate_vrd_annos(DATASET, str(DATA_PATH))
    print(f'[SKIP] Uncomment above line to run. Output: data/metadata/{DATASET}_annotations.pkl')
finally:
    _os.chdir(_orig_cwd)

In [ ]:
# Step B: Generate frame indices for training
# This creates data/metadata/<db>_train_frames_stage1.json (and stage2)

_os.chdir(str(ROOT_DIR / 'data'))
try:
    # Stage 1 frame indices (pairs of frames)
    # get_trainval_frameids(DATASET, str(ROOT_DIR), 'train', stage=1, timestep=1, minmax_dur=24)
    # Stage 2 frame indices (clips of 8 frames)
    # get_trainval_frameids(DATASET, str(ROOT_DIR), 'train', stage=2, timestep=1, minmax_dur=24)
    # Validation frame indices
    # get_trainval_frameids(DATASET, str(ROOT_DIR), 'val', stage=2, timestep=1, minmax_dur=24)
    print(f'[SKIP] Uncomment lines above to run.')
    print(f'Output artifacts:')
    print(f'  data/metadata/{DATASET}_train_frames_stage1.json')
    print(f'  data/metadata/{DATASET}_train_frames_stage2.json')
    print(f'  data/metadata/{DATASET}_val_frames.json')
finally:
    _os.chdir(_orig_cwd)

## 5. Inspect the Generated Annotation Pickle

The pickle maps `video_id -> {frame_annos: {fid: {...}}, rel_tag_uids: [...]}`.

In [ ]:
METADATA_DIR = ROOT_DIR / 'data' / 'metadata'
anno_pickle_path = METADATA_DIR / f'{DATASET}_annotations.pkl'

if anno_pickle_path.exists():
    with open(anno_pickle_path, 'rb') as f:
        annotations = pickle.load(f)
    
    print(f'Loaded {len(annotations)} videos from {anno_pickle_path}')
    sample_vid = list(annotations.keys())[0]
    
    print(f'\nTop-level keys per video: {list(annotations[sample_vid].keys())}')
    print(f'Number of annotated frames: {len(annotations[sample_vid]["frame_annos"])}')
    
    # Pick one frame and inspect
    frame_annos = annotations[sample_vid]['frame_annos']
    sample_fid = list(frame_annos.keys())[0]
    frame_data = frame_annos[sample_fid]
    
    print(f'\nFrame {sample_fid} annotation:')
    for key, value in frame_data.items():
        if isinstance(value, (list, np.ndarray)):
            print(f'  {key}: type={type(value).__name__}, len={len(value)}')
            if len(value) > 0:
                print(f'    first element: {value[0]}')
        else:
            print(f'  {key}: {value}')
else:
    print(f'Annotation pickle not found at {anno_pickle_path}')
    print('Run the data preparation steps above first.')

### Frame annotation structure

Each frame's data contains:

| Key | Type | Description |
|-----|------|-------------|
| `sub_labels` | `list[int]` | Object class index for each subject |
| `obj_labels` | `list[int]` | Object class index for each object |
| `verb_labels` | `list[list[int]]` | Multi-label predicate IDs (one-hot later) |
| `sub_boxes` | `list[[x1,y1,x2,y2]]` | Subject bounding boxes (xyxy, pixel coords) |
| `obj_boxes` | `list[[x1,y1,x2,y2]]` | Object bounding boxes (xyxy, pixel coords) |
| `so_track_ids` | `list[[sub_tid, obj_tid]]` | Track IDs linking subjects & objects across frames |

**The number of instances (N) varies per frame** — each frame can have 0 to many concurrent relation instances.

## 6. Inspect the Frame Index (Training Clips)

The frame index JSON defines which frame sequences form valid training clips.

In [ ]:
# Check stage 1 frame index
stage1_json = METADATA_DIR / f'{DATASET}_train_frames_stage1.json'
stage2_json = METADATA_DIR / f'{DATASET}_train_frames_stage2.json'
val_json = METADATA_DIR / f'{DATASET}_val_frames.json'

for path, name in [(stage1_json, 'Stage 1 Train'), (stage2_json, 'Stage 2 Train'), (val_json, 'Validation')]:
    if path.exists():
        with open(path, 'r') as f:
            data = json.load(f)
        if 'train_begin_fids' in data:
            print(f'{name}: {len(data["train_begin_fids"])} training clips')
            print(f'  Example: video={data["train_begin_fids"][0]}, duration={data["durations"][0]}')
        else:
            print(f'{name}: {len(data)} videos with frame bitmaps')
            sample = list(data.values())[0]
            print(f'  Example frame bitmap (first 20): {sample[:20]}')
    else:
        print(f'{name}: NOT FOUND at {path}')

## 7. Visualize Annotations on Video Frames

Let's load a video with decord and draw bounding boxes for subject-object pairs with relation labels.

In [ ]:
# Check if decord is available
try:
    from decord import VideoReader, cpu
    print('decord available')
except ImportError:
    print('decord not installed. Install with: pip install decord')
    print('Falling back to OpenCV for video reading...')
    import cv2

In [ ]:
# Find a sample video and visualize annotations
video_dir = DATA_PATH / 'videos'
if video_dir.exists() and anno_pickle_path.exists():
    # Pick first video that has annotations
    sample_vid = list(annotations.keys())[0]
    
    # Look for the video file
    video_path = video_dir / f'{sample_vid}.mp4'
    if not video_path.exists():
        # Try alternative naming
        candidates = list(video_dir.glob(f'{sample_vid}.*'))
        if candidates:
            video_path = candidates[0]
    
    print(f'Video: {sample_vid}')
    print(f'Path: {video_path}')
    print(f'Exists: {video_path.exists()}')
    
    if video_path.exists():
        # Load video with decord
        vr = VideoReader(str(video_path), ctx=cpu(0))
        print(f'Total frames: {len(vr)}')
        
        # Pick a frame with annotations
        frame_annos = annotations[sample_vid]['frame_annos']
        fids_with_annos = sorted([int(k) for k in frame_annos.keys()])
        print(f'Frames with annotations: {len(fids_with_annos)}')
        
        # Show first annotated frame
        fid = fids_with_annos[0]
        frame = vr[fid].asnumpy()  # (H, W, 3) RGB
        
        # Load annotations for this frame
        ann = frame_annos[str(fid)] if str(fid) in frame_annos else frame_annos[fid]
        
        # Map class indices to names
        obj_list = vidvrd_objects if DATASET == 'vidvrd' else vidor_objects
        pred_list = vidvrd_predicates if DATASET == 'vidvrd' else vidor_relations
        
        # Draw boxes
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        ax.imshow(frame)
        
        colors = plt.cm.tab20(np.linspace(0, 1, len(ann['sub_boxes'])))
        for i, (sbox, obox, slabel, olabel, vlabels) in enumerate(zip(
            ann['sub_boxes'], ann['obj_boxes'], 
            ann['sub_labels'], ann['obj_labels'], 
            ann['verb_labels']
        )):
            # Subject box (red)
            rect_s = patches.Rectangle(
                (sbox[0], sbox[1]), sbox[2]-sbox[0], sbox[3]-sbox[1],
                linewidth=2, edgecolor='red', facecolor='none'
            )
            ax.add_patch(rect_s)
            ax.text(sbox[0], sbox[1]-5, obj_list[slabel], color='red', fontsize=10, fontweight='bold')
            
            # Object box (green)
            rect_o = patches.Rectangle(
                (obox[0], obox[1]), obox[2]-obox[0], obox[3]-obox[1],
                linewidth=2, edgecolor='green', facecolor='none'
            )
            ax.add_patch(rect_o)
            ax.text(obox[0], obox[1]-5, obj_list[olabel], color='green', fontsize=10, fontweight='bold')
            
            # Predicate labels
            pred_names = [pred_list[pid] for pid, flag in enumerate(vlabels) if flag]
            mid_x = (sbox[0] + obox[0]) / 2
            mid_y = (sbox[1] + obox[1]) / 2
            ax.text(mid_x, mid_y, ', '.join(pred_names), 
                    color='yellow', fontsize=8, bbox=dict(facecolor='black', alpha=0.5))
        
        ax.set_title(f'{sample_vid} - Frame {fid} ({len(ann["sub_boxes"])} relations)')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Video directory or annotations not found.')
    print(f'Video dir: {video_dir} (exists: {video_dir.exists() if "video_dir" in dir() else "N/A"})')

## 8. Understanding the Two-Stage Data Split

VRDFormer uses a **two-stage training paradigm**. The same underlying annotations serve both stages, but the data loading differs:

| Aspect | Stage 1 (Detection) | Stage 2 (Classification) |
|--------|---------------------|--------------------------|
| **Frames per sample** | 2 (current + previous) | 8 (full clip) |
| **Minimum clip duration** | 2 frames | 4 frames + class consistency |
| **Query source** | Learned embeddings + track queries | ROI Align from GT boxes |
| **Loss** | Per-frame: cls + bbox L1 + GIoU | Per-clip: relation classification |
| **Batch size** | 4 (multi-GPU) | 1 (single GPU) |
| **Tracking** | Yes (previous frame propagated) | No (GT track IDs used directly) |

**Why two stages?** Stage 1 learns to detect subject-object pairs and their spatial relationships from individual frames. Stage 2 then uses the detected boxes to classify the temporal relationship (predicate) by aggregating features across the full video clip.

## 9. Dataset Statistics

Let's compute some useful statistics about the dataset.

In [ ]:
if anno_pickle_path.exists():
    # Count total relations, frames, etc.
    total_frames = 0
    total_relations = 0
    predicates_per_frame = []
    objects_per_frame = []
    
    for vid, data in tqdm(annotations.items(), desc='Computing stats'):
        for fid, ann in data['frame_annos'].items():
            n = len(ann['sub_labels'])
            total_relations += n
            total_frames += 1
            predicates_per_frame.append(n)
            objects_per_frame.append(len(set(ann.get('sub_track_ids', [])) | set(ann.get('obj_track_ids', []))))
    
    print(f'=== {DATASET.upper()} Dataset Statistics ===')
    print(f'Total videos: {len(annotations)}')
    print(f'Total annotated frames: {total_frames}')
    print(f'Total relation instances: {total_relations}')
    print(f'Avg relations per frame: {total_relations/max(1,total_frames):.2f}')
    print(f'Avg unique objects per frame: {np.mean(objects_per_frame):.2f}')
    print(f'Frames with no relations: {predicates_per_frame.count(0)}')
    print(f'Max relations in a single frame: {max(predicates_per_frame)}')
else:
    print(f'Run data preparation first to generate {anno_pickle_path}')

## 10. Summary

**What we've learned:**

1. **VidVRD** has 35 object classes, 132 predicates. **VidOR** has 80 objects, 50 predicates (42 actions + 8 spatial).
2. **Raw annotations** are per-video JSONs with object trajectories (bbox per frame + track ID) and relation instances (subject_tid, object_tid, predicate, time span).
3. **Data preparation** (`data/prepare.py`):
   - `generate_vrd_annos()`: Converts per-video JSONs → per-frame pickle with normalized boxes, class indices, track IDs, verb labels
   - `get_trainval_frameids()`: Creates frame index JSONs defining valid training clips
4. **Two stages** use the same annotations differently: Stage 1 loads frame pairs for detection, Stage 2 loads 8-frame clips for relation classification.

**Next:** Notebook 2 will show how the PyTorch Dataset and DataLoader turn this into model-ready tensors.